In [0]:
from datetime import date

dbutils.widgets.dropdown(
    "execution_mode",
    "backfill",
    ["backfill", "daily"]
)

dbutils.widgets.text("start_date", "2018-10-18")
dbutils.widgets.text("end_date", str(date.today()))
dbutils.widgets.text("orders_per_day", "100")

execution_mode = dbutils.widgets.get("execution_mode")
start_date = dbutils.widgets.get("start_date")
end_date = dbutils.widgets.get("end_date")
orders_per_day = int(
    dbutils.widgets.get("orders_per_day")
)

catalog = "workspace"
schema = "olist_landing"


if orders_per_day <= 0:
    raise ValueError("orders_per_day deve ser maior que zero")

if execution_mode == "daily":
    start_date = str(date.today())
    end_date = start_date

start_dt = date.fromisoformat(start_date)
end_dt = date.fromisoformat(end_date)

if start_dt > end_dt:
    raise ValueError("start_date não pode ser maior que end_date")

total_days = (end_dt - start_dt).days + 1
total_orders = total_days * orders_per_day


print(f"Execution mode: {execution_mode}")
print(f"Start date: {start_date}")
print(f"End date: {end_date}")
print(f"Orders per day: {orders_per_day}")
print({total_days})
print({total_orders})

In [0]:
from pyspark.sql import functions as F

products_sellers_df = (
    spark.table("workspace.olist_landing.bronze_items")
    .select("product_id", "seller_id", "price", "freight_value")
    .filter(
        F.col("product_id").isNotNull()
        & F.col("seller_id").isNotNull()
        & (F.col("price") > 0)
        & (F.col("freight_value") >= 0)
    )
    .dropDuplicates(["product_id", "seller_id", "price", "freight_value"])
)

display(products_sellers_df.limit(10))

In [0]:
customers_df = (
    spark.table("workspace.olist_landing.bronze_customers")
    .select("customer_id", "customer_unique_id")
    .dropDuplicates(["customer_id"])
    .filter(
        F.col("customer_id").isNotNull()
        & F.col("customer_unique_id").isNotNull()
    )
)

display(customers_df.limit(10))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from uuid import uuid4

run_id = uuid4().hex

base_orders_df = (
    spark.range(total_orders)
    .withColumn(
        "day_offset",
        F.floor(F.col("id") / orders_per_day).cast("int")
    )
    .withColumn(
        "purchase_date",
        F.date_add(F.lit(start_date), F.col("day_offset"))
    )
)


customer_pool_df = (
    customers_df
    .select("customer_id")
    .dropDuplicates()
    .withColumn(
        "customer_rn",
        F.row_number().over(
            Window.orderBy("customer_id")
        )
    )
)

customer_count = customer_pool_df.count()

if customer_count == 0:
    raise ValueError("Nenhum cliente disponível")


new_orders_df = (
    base_orders_df
    .withColumn(
        "customer_rn",
        (F.col("id") % customer_count + 1).cast("long")
    )
    .join(customer_pool_df, "customer_rn")
    .withColumn(
        "order_id",
        F.sha2(
            F.concat_ws(
                "-",
                F.lit(run_id),
                F.col("id").cast("string")
            ),
            256
        )
    )
    .withColumn(
        "order_status",
        F.lit("approved")
    )
    .withColumn(
        "order_purchase_timestamp",
        F.expr(
            "CAST(purchase_date AS TIMESTAMP) + INTERVAL 8 HOURS"
        )
    )
    .withColumn(
        "order_approved_at",
        F.expr(
            "order_purchase_timestamp + INTERVAL 15 MINUTES"
        )
    )
    .withColumn(
        "order_delivered_carrier_date",
        F.lit(None).cast("timestamp")
    )
    .withColumn(
        "order_delivered_customer_date",
        F.lit(None).cast("timestamp")
    )
    .withColumn(
        "order_estimated_delivery_date",
        F.expr(
            "order_purchase_timestamp + INTERVAL 10 DAYS"
        )
    )
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    )
)

display(new_orders_df.limit(100))

In [0]:

products_df = (
    spark.table("workspace.olist_landing.bronze_items")
    .select("product_id", "seller_id", "price", "freight_value")
    .dropDuplicates(["product_id", "seller_id", "price", "freight_value"])
    .filter(
        F.col("product_id").isNotNull()
        & F.col("seller_id").isNotNull()
        & F.col("price").isNotNull()
        & F.col("freight_value").isNotNull()
    )
)

sample_products_df = (
    products_df
    .orderBy(F.rand(seed=42))
    .limit(10)
    .withColumn(
        "rn",
        F.row_number().over(Window.orderBy("product_id", "seller_id", "price", "freight_value"))
    )
)

product_count = sample_products_df.count()

orders_numbered_df = (
    new_orders_df
    .select("order_id", "order_purchase_timestamp")
    .withColumn(
        "order_rn",
        F.row_number().over(Window.orderBy("order_id"))
    )
    .withColumn(
        "rn",
        ((F.col("order_rn") - 1) % product_count + 1).cast("int")
    )
)

new_items_df = (
    orders_numbered_df
    .join(sample_products_df, "rn")
    .select(
        "order_id",
        F.lit(1).alias("order_item_id"),
        "product_id",
        "seller_id",
        F.expr(
            "order_purchase_timestamp + INTERVAL 7 DAYS"
        ).alias("shipping_limit_date"),
        F.col("price").cast("double").alias("price"),
        F.col("freight_value").cast("double").alias("freight_value")
    )
)

display(new_items_df)

In [0]:

from pyspark.sql.window import Window

payments_df = (
    spark.table("workspace.olist_landing.bronze_payments")
    .select("payment_type", "payment_installments")
    .filter(
        F.col("payment_type").isNotNull()
        & F.col("payment_installments").isNotNull()
        & (F.col("payment_type") != "not_defined")
        & (
            (
                (F.col("payment_type") == "credit_card")
                & (F.col("payment_installments") >= 1)
            )
            |
            (
                (F.col("payment_type") != "credit_card")
                & (F.col("payment_installments") == 1)
            )
        )
    )
)

payment_history = (
    payments_df
    .groupBy("payment_type")
    .agg(
        F.count("*").alias("total_per_type")
    )
)

total_payments = payments_df.count()

payment_distribution_df = (
    payment_history
    .withColumn(
        "percentage",
        F.round(
            (F.col("total_per_type") / F.lit(total_payments)) * 100,
            2
        )))
        
window_spec = Window.orderBy(
    F.desc("total_per_type"),
    F.asc("payment_type")
)

payment_probability_df = (
    payment_distribution_df
    .withColumn(
        "probability",
        F.col("total_per_type") / F.lit(total_payments)
    )
    .withColumn(
        "cumulative_probability",
        F.sum("probability").over(window_spec)
    )
)
orders_random_df = (
    new_orders_df
    .select("order_id")
    .withColumn("random_value", F.rand(seed=42))
)

payment_ranges_df = (
    payment_probability_df
    .withColumn(
        "lower_bound",
        F.col("cumulative_probability") - F.col("probability")
    ))

orders_with_payment_type_df = (
    orders_random_df
    .crossJoin(
        F.broadcast(
            payment_ranges_df.select(
                "payment_type",
                "lower_bound",
                "cumulative_probability"
            )))
    .filter(
        (F.col("random_value") >= F.col("lower_bound"))
        & (F.col("random_value") < F.col("cumulative_probability"))
    )
    .select("order_id", "payment_type"))

display(orders_with_payment_type_df.groupBy("payment_type").count())
print("orders:", new_orders_df.count())
print("order with payment:", orders_with_payment_type_df.count())


In [0]:
credit_installments_df = (
    payments_df
    .filter(F.col("payment_type") == "credit_card")
    .groupBy("payment_installments")
    .agg(
        F.count("*").alias("total")
    )
    .orderBy("payment_installments")
)

total_credit = (
    credit_installments_df
    .agg(F.sum("total").alias("total"))
    .first()["total"]
)

if total_credit is None or total_credit == 0:
    raise ValueError("Nenhum pagamento com cartão de crédito encontrado")


window_installments = (
    Window.orderBy("payment_installments")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    ))


installments_probability_df = (
    credit_installments_df
    .withColumn(
        "probability",
        F.col("total") / F.lit(total_credit)
    )
    .withColumn(
        "cumulative_probability",
        F.sum("probability").over(window_installments)
    ))


installments_ranges_df = (
    installments_probability_df
    .withColumn(
        "lower_bound",
        F.col("cumulative_probability") - F.col("probability")
    )
    .select(
        "payment_installments",
        "lower_bound",
        "cumulative_probability"
    )
)


credit_orders_df = (
    orders_with_payment_type_df
    .filter(F.col("payment_type") == "credit_card")
    .withColumn("random_value", F.rand(seed=123))
)

credit_orders_installments_df = (
    credit_orders_df
    .crossJoin(F.broadcast(installments_ranges_df))
    .filter(
        (F.col("random_value") >= F.col("lower_bound"))
        & (F.col("random_value") < F.col("cumulative_probability"))
    )
    .select(
        "order_id",
        "payment_type",
        "payment_installments"
    )
)

other_orders_installments_df = (
    orders_with_payment_type_df
    .filter(F.col("payment_type") != "credit_card")
    .withColumn("payment_installments", F.lit(1))
    .select(
        "order_id",
        "payment_type",
        "payment_installments"
    )
)

orders_with_installments_df = (
    credit_orders_installments_df
    .unionByName(other_orders_installments_df)
)

order_values_df =  (new_items_df
                    .groupBy("order_id")
                    .agg(
                        F.round(
                            F.sum(
                                F.col("price")+ F.col("freight_value")
                            ),2
                        ).alias("payment_value")

                    )            
)
new_payments_df = (
    orders_with_installments_df
    .join(
        order_values_df,
        on="order_id",
        how="inner"
    )
    .withColumn(
        "payment_sequential",
        F.lit(1)
    )
    .select(
        "order_id",
        "payment_sequential",
        "payment_type",
        "payment_installments",
        "payment_value"
    )
)

display(new_payments_df.limit(100))

In [0]:
# Validation
null_validation_df = new_payments_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in new_payments_df.columns
])

invalid_payments_df = (
    new_payments_df
    .filter(F.col("payment_value") <= 0)
)

invalid_pament_type_df= (
           new_payments_df 
           .filter(
               (F.col("payment_type").isin("boleto","voucher", "debit_card")) & (F.col("payment_installments") != 1)

           ) 
)

invalid_crediCard = (
        new_payments_df
        .filter(
            (F.col("payment_type") == "credit card") & (F.col("payment_installments") < 1)
        )
)


display(invalid_pament_type_df)
display(null_validation_df)
display(invalid_payments_df)
display(invalid_crediCard)

In [0]:
batch_path = "/Volumes/workspace/olist_landing/batched_files/batch_streaming"

orders_path = f"{batch_path}/orders"
items_path = f"{batch_path}/items"
payments_path = f"{batch_path}/payments"

for path in [orders_path, items_path, payments_path]:
    dbutils.fs.mkdirs(path)
    print(f"folder created: {path}")



In [0]:
from uuid import uuid4

run_id = uuid4().hex
temp_path = f"{batch_path}/_temp_orders_{run_id}"
final_path = f"{orders_path}/orders_{run_id}.csv"

(
    new_orders_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(temp_path)
)

csv_file = next(
    f.path for f in dbutils.fs.ls(temp_path)
    if f.name.startswith("part-") and f.name.endswith(".csv")
)

dbutils.fs.mv(csv_file, final_path)
dbutils.fs.rm(temp_path, recurse=True)



In [0]:
temp_items_path = f"{batch_path}/_temp_items_{run_id}"
final_items_path = f"{items_path}/items_{run_id}.csv"

(
    new_items_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(temp_items_path)
)

csv_file = next(
    f.path
    for f in dbutils.fs.ls(temp_items_path)
    if f.name.startswith("part-") and f.name.endswith(".csv")
)

if any(f.path == final_items_path for f in dbutils.fs.ls(items_path)):
    raise ValueError(f"Arquivo já existe: {final_items_path}")

dbutils.fs.mv(csv_file, final_items_path)
dbutils.fs.rm(temp_items_path, recurse=True)


In [0]:
temp_payments_path = f"{batch_path}/_temp_payments_{run_id}"
final_payments_path = f"{payments_path}/payments_{run_id}.csv"
(
    new_payments_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(temp_payments_path)
)

csv_file = next(
    f.path
    for f in dbutils.fs.ls(temp_payments_path)
    if f.name.startswith("part-") and f.name.endswith(".csv")
)

if any(f.path == final_payments_path for f in dbutils.fs.ls(payments_path)):
    raise ValueError(f"Arquivo já existe: {final_payments_path}")

dbutils.fs.mv(csv_file, final_payments_path)
dbutils.fs.rm(temp_payments_path, recurse=True)

print("Arquivo de pagamentos gravado:", final_payments_path)

In [0]:
#Validation

batch_path = f"/Volumes/{catalog}/{schema}/batched_files/batch_streaming"
for entity in ["orders", "items", "payments"]:
    print(f"\n{'=' * 40}")
    print(f"validating: {entity.upper()}")
    print(f"{'=' * 40}")
    csv_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{batch_path}/{entity}/*.csv")
    )

    bronze_df = spark.table(
        f"{catalog}.{schema}.bronze_{entity}"
    )
    bronze_columns = [
        c for c in bronze_df.columns
        if not c.startswith("_")
    ]
    csv_columns = csv_df.columns

    print("CSV Columns:", csv_columns)
    print("Bronze Columns:", bronze_columns)
    print(
        "CSV Missing Columns:",
        set(bronze_columns) - set(csv_columns)
    )

    print(
        "CSV Extras Coluns:",
        set(csv_columns) - set(bronze_columns)
    )

In [0]:
for entity in ["orders", "items", "payments"]:

    print(f"\n===== {entity.upper()} =====")

    csv_df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(f"{batch_path}/{entity}/*.csv")
    )

    bronze_df = spark.table(
        f"{catalog}.{schema}.bronze_{entity}"
    )

    csv_types = dict(csv_df.dtypes)
    bronze_types = dict(bronze_df.dtypes)

    for column in csv_df.columns:
        csv_type = csv_types[column]
        bronze_type = bronze_types.get(column)

        status = "OK" if csv_type == bronze_type else "Check"

        print(
            f"{column}: "
            f"CSV={csv_type} | "
            f"Bronze={bronze_type} | "
            f"{status}"
        )

In [0]:
from pyspark.sql.functions import col, count, when, trim
orders_csv = (
    spark.read
    .option("header", "true")
    .csv(
        "/Volumes/workspace/olist_landing/"
        "batched_files/batch_streaming/orders/*.csv"
    )
)
orders_csv.select(
    *[
        count(
            when(
                col(c).isNotNull() & (trim(col(c)) != ""),
                1
            )
        ).alias(c)
        for c in [
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ]
).show()